# Open Jev Starter Kit & Benchmark Explorer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LambertLeong/LambertLeong.github.io/blob/master/assets/notebooks/open-decision-models/open-jev-starter-kit.ipynb)

**Route one message. Inspect the evidence. Compare generation with a direct readout.**

Start with the benchmark explorer if you only want the results: it needs no model downloads. The playground uses the official AnyJev L0 implementation; a later section contrasts two readouts of the same pinned Qwen model, and the final section can re-run the full eight-configuration benchmark on a GPU.

The accepted eight-configuration results are embedded, so the explorer works offline. Qwen JSON generation reached 83% fixed-route accuracy and 66% changing-candidate accuracy. Nine dynamic responses failed the strict format policy because they wrapped JSON in markdown fences.

## Setup

Install once, then restart the runtime if Colab asks. Model downloads and installation are separate from inference timing. A GPU runtime (Runtime → Change runtime type → T4) is faster; CPU also works but is slower. Model downloads on a fresh runtime take a few minutes.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'anyjev[hf] @ git+https://github.com/nokia-applied-research/AnyJev.git@f82fe03bc0d01c9e340b4f8880b744e78ea6116d',
    'transformers==4.57.6', 'plotly', 'ipywidgets', 'pandas'])

## 60-second playground

AnyJev reads candidate scores from a language model without generating an answer string. Load the pinned checkpoint once, then edit the request and options below.

In [ ]:
import torch
from anyjev import Decider, Question
from anyjev.backends.hf import HFBackend
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CHECKPOINT = 'Qwen/Qwen3-0.6B'
REVISION = 'c1899de289a04d12100db370d81485cdf75e47ca'
backend = HFBackend(CHECKPOINT, revision=REVISION, device=DEVICE, dtype='float32', batch_size=1)
model = Decider(backend, level='L0')

In [ ]:
query = "I was charged twice for the same transfer."
candidates = ["duplicate charge", "transfer delay", "cash withdrawal", "close account"]
decision = model.decide(query, [Question.choice("Choose the banking route.", candidates, name='route')])['route']
print('Chosen route:', decision.argmax, '\nProbabilities:', decision.distribution)

## Interactive benchmark explorer

These are the accepted GPU results: 200 fixed-route requests and 100 changing-candidate requests, repeated twice. Select a panel to compare accuracy. This view does not rank speed across SDKs with different timing boundaries.

The cell tries the public comparison CSVs first and falls back to the embedded accepted data when the files are unpublished or unavailable. All eight configurations are included.

In [ ]:
import json, urllib.request
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display
EMBEDDED_ROWS = [{'project_id': 'kev', 'configuration': 'Kev', 'panel': 'fixed', 'accuracy': 0.98, 'macro_f1': 0.9800902668759812}, {'project_id': 'laya', 'configuration': 'Laya', 'panel': 'fixed', 'accuracy': 0.92, 'macro_f1': 0.9154526048478205}, {'project_id': 'verdict', 'configuration': 'Verdict', 'panel': 'fixed', 'accuracy': 0.865, 'macro_f1': 0.8700761172854488}, {'project_id': 'qwen_generative', 'configuration': 'Qwen3-0.6B Direct JSON Generation', 'panel': 'fixed', 'accuracy': 0.83, 'macro_f1': 0.8104055692226367}, {'project_id': 'anyjev', 'configuration': 'AnyJev L0 + Qwen3-0.6B', 'panel': 'fixed', 'accuracy': 0.795, 'macro_f1': 0.7714449190528925}, {'project_id': 'simple_jev', 'configuration': 'Simple Jev + Qwen3-0.6B', 'panel': 'fixed', 'accuracy': 0.79, 'macro_f1': 0.7634907702044669}, {'project_id': 'openjev', 'configuration': 'OpenJev + Qwen3-0.6B', 'panel': 'fixed', 'accuracy': 0.79, 'macro_f1': 0.7465899005925587}, {'project_id': 'nanojev', 'configuration': 'NanoJev', 'panel': 'fixed', 'accuracy': 0.52, 'macro_f1': 0.48567585691422444}, {'project_id': 'kev', 'configuration': 'Kev', 'panel': 'dynamic', 'accuracy': 0.98, 'macro_f1': 0.989795918367347}, {'project_id': 'verdict', 'configuration': 'Verdict', 'panel': 'dynamic', 'accuracy': 0.89, 'macro_f1': 0.9288226654298084}, {'project_id': 'laya', 'configuration': 'Laya', 'panel': 'dynamic', 'accuracy': 0.74, 'macro_f1': 0.7480376766091051}, {'project_id': 'qwen_generative', 'configuration': 'Qwen3-0.6B Direct JSON Generation', 'panel': 'dynamic', 'accuracy': 0.66, 'macro_f1': 0.7384167548201669}, {'project_id': 'openjev', 'configuration': 'OpenJev + Qwen3-0.6B', 'panel': 'dynamic', 'accuracy': 0.6, 'macro_f1': 0.7032193996479711}, {'project_id': 'simple_jev', 'configuration': 'Simple Jev + Qwen3-0.6B', 'panel': 'dynamic', 'accuracy': 0.58, 'macro_f1': 0.6904682715658325}, {'project_id': 'anyjev', 'configuration': 'AnyJev L0 + Qwen3-0.6B', 'panel': 'dynamic', 'accuracy': 0.56, 'macro_f1': 0.6308761430953074}, {'project_id': 'nanojev', 'configuration': 'NanoJev', 'panel': 'dynamic', 'accuracy': 0.17, 'macro_f1': 0.24880593262946205}]
EMBEDDED_ERRORS = [{'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Assist me please with card activation.', 'target': 'activate_my_card', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'How can I change my Tholepin ?', 'target': 'change_pin', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'My new pin needs to be set.', 'target': 'change_pin', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Do cash machines let me change my pin?', 'target': 'change_pin', 'prediction': 'activate_my_card', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'My card payment is still on hold, why is it so?', 'target': 'pending_card_payment', 'prediction': 'activate_my_card', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Why is my payment pending?', 'target': 'pending_card_payment', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'How do i finalize a card payment', 'target': 'pending_card_payment', 'prediction': 'activate_my_card', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Why has my payment not gone through?', 'target': 'pending_card_payment', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'When will my card payment be done pending?', 'target': 'pending_card_payment', 'prediction': 'activate_my_card', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': "I made this payment days ago so why hasn't it gone through yet?", 'target': 'pending_card_payment', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'I am waiting for a payment to go through.', 'target': 'pending_card_payment', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'How long does an item I bought show as pending?', 'target': 'pending_card_payment', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Why is there a pending transaction on my card?', 'target': 'pending_card_payment', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'How long do card transactions take to clear?', 'target': 'pending_card_payment', 'prediction': 'activate_my_card', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'what does pending mean?', 'target': 'pending_card_payment', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Are there any fees for adding money with an international card?', 'target': 'top_up_by_card_charge', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'i was charged when i used a us issued card. why and what cards are free to use to add money', 'target': 'top_up_by_card_charge', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Any hidden fees associated with the international card if money is added?', 'target': 'top_up_by_card_charge', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Will I get charged for topping off my card?', 'target': 'top_up_by_card_charge', 'prediction': 'activate_my_card', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Are there charges for topping up US cards?', 'target': 'top_up_by_card_charge', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Are there fees for top ups?', 'target': 'top_up_by_card_charge', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'What are the charges for US cards with top up.', 'target': 'top_up_by_card_charge', 'prediction': 'activate_my_card', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Is there an additional charge for topping up using a European card?', 'target': 'top_up_by_card_charge', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'In exchange for top ups will you take fees?', 'target': 'top_up_by_card_charge', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'I just topped off my card will I be charged for it?', 'target': 'top_up_by_card_charge', 'prediction': 'activate_my_card', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Are there any fees if I use a European bank card for a top up?', 'target': 'top_up_by_card_charge', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'What are the top-up fees?', 'target': 'top_up_by_card_charge', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Will there be any extra fees for European bank card for top up?', 'target': 'top_up_by_card_charge', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Will I be charged if I use European bank card for top up?', 'target': 'top_up_by_card_charge', 'prediction': 'activate_my_card', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'I want to use a European bank card for a top up. Must I pay?', 'target': 'top_up_by_card_charge', 'prediction': 'activate_my_card', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Will I be charged for topping up by card?', 'target': 'top_up_by_card_charge', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Is there a fee for topping up', 'target': 'top_up_by_card_charge', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Will it cost anything to top up a US card?', 'target': 'top_up_by_card_charge', 'prediction': 'activate_my_card', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'The app denied my topped up.', 'target': 'top_up_failed', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': "Heyy I tried to top up with my card today but it failed! I just got the couple days before but it has worked last time. What's going on, can you double check please?", 'target': 'top_up_failed', 'prediction': 'top_up_by_card_charge', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': "I have a top up that didn't go through.  Why?", 'target': 'top_up_failed', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': "So, I tried topping up my card for the today. Unfortunately, it failed. A couple of days back I used it and it works fine. Can you double check and tell me what's going on?", 'target': 'top_up_failed', 'prediction': 'activate_my_card', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Why did the app deny my top up?', 'target': 'top_up_failed', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': "My top-up isn't working.", 'target': 'top_up_failed', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'I attempted to top up but the app denied it.', 'target': 'top_up_failed', 'prediction': 'top_up_by_card_charge', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'What reason did my top-up fail for?', 'target': 'top_up_failed', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Why didn`t my topup go through?', 'target': 'top_up_failed', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'How can I check to see if my top-up worked?', 'target': 'top_up_failed', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': "My top up didn't work.", 'target': 'top_up_failed', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'My top up failed.', 'target': 'top_up_failed', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Why would my topup not go through?', 'target': 'top_up_failed', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Why caused my top up to not work?', 'target': 'top_up_failed', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Why did my top up not work?', 'target': 'top_up_failed', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Why was my top-up unsuccessful?', 'target': 'top_up_failed', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': "My funding to my card didn't go through.", 'target': 'top_up_failed', 'prediction': 'activate_my_card', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'I need to know why my credit card was declined for top up? What is the deal here and why did it not go through?', 'target': 'top_up_failed', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'I topped up but the app did not accept it.', 'target': 'top_up_failed', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'The app denied my top-up', 'target': 'top_up_failed', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'I think my top up did not work', 'target': 'top_up_failed', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'My top up was denied in the app.', 'target': 'top_up_failed', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Can you tell me if my pop-up went through?', 'target': 'top_up_failed', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': "can you check why I wasn't able to top up?", 'target': 'top_up_failed', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': "I'd rather not verify my identity.", 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'do I need to verify my identity every time I log into my account?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Why am I required to verify my identity?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Can I make transfers before identity verification?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'What is the need to verify my identity?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Do I have to do the identity check?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Can I use my account without verifying my identity?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Do I have to do an identity check?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Can I still use my account, even though the identity verification has not passed yet?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'What is the purpose for the identity check', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': "I don't like that I have to fill out so much information about my identity.", 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'What is the point of an identity check?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'How come you need to know my identity?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'May I use my account now, even though the identity verification has not gone through yet?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Do I need to verify my identity to use my account?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Can i make transactions before identity verification is complete?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Do I really need to verify my identity?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': "I did not get the right amount of money when I used a different country's currency.", 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'I took out money from a transaction machine and it exchanged the wrong dollar value amount from another currency!', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': "I made a withdraw from the ATM this past holiday and it seems like a was charged too much. If I would have   known about these charges I wouldn't have made a withdraw.", 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Can you look to make sure the exchange rate is correct', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Exchange rate for my cash withdrawal is wrong', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': "Hello. I'm on holiday and didn't bring any cash with me. I need to withdrawal my home currency from one of your machines. Do you have any that will do this and is there a charge?", 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'I attempted to get money using a foreign currency at an ATM but the rate was highly inaccurate!', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Why is the exchange rate wrong for my international withdrawal?', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'A wrong exchange rate was applied to a transaction made abroad.', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'The exchange rate you gave me for my cash withdrawal is wrong', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Is there a fee for exchanging cash?', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Why is the exchange rate so exorbitant? This should have been a much higher amount of cash for that to apply.', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Why was the exchange rate different when I withdrew my cash?', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Will there be additional costs of I make a withdrawal from a local ATM of British pounds? I need some cash to feel comfortable on the journey home', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'It seems I was overcharged when I used an ATM while on vacation. If I knew about your fees in advance I sure would have gone somewhere else.', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'I checked the exchange rate when I withdrew cash. But the actual rate that you applied was different.', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'In receiving cash, the wrong exchange rate was used for my transaction.', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'Can I make a withdraw if I am out of country from with out any fees at an ATM?', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'I feel like too much money was taken during my currency exchange.', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'There was an error in exchange rate for my cash withdrawal', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'I need the right foreign money, at the right rate, at ATMs, when I go abroad.', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'fixed', 'message': 'The conversion rate on the ATM that I used with foreign currency was wrong.', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'I have a strange payment in my statement', 'target': 'card_payment_not_recognised', 'prediction': 'why_verify_identity', 'candidates': ['why_verify_identity', 'card_payment_not_recognised', 'top_up_failed', 'pending_card_payment'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'There is a charge on my card that I did not make', 'target': 'card_payment_not_recognised', 'prediction': 'activate_my_card', 'candidates': ['activate_my_card', 'wrong_exchange_rate_for_cash_withdrawal', 'pending_card_payment', 'card_payment_not_recognised'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'I saw a payment i did not do', 'target': 'card_payment_not_recognised', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'verify_my_identity', 'pending_card_payment', 'card_payment_not_recognised'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'There is a payment that is not mine in the app.  Please advise/', 'target': 'card_payment_not_recognised', 'prediction': 'why_verify_identity', 'candidates': ['card_payment_not_recognised', 'why_verify_identity', 'top_up_by_card_charge', 'contactless_not_working'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'An unauthorized payment is in my app', 'target': 'card_payment_not_recognised', 'prediction': 'verify_my_identity', 'candidates': ['verify_my_identity', 'contactless_not_working', 'change_pin', 'card_payment_not_recognised'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'I did not make this card payment.', 'target': 'card_payment_not_recognised', 'prediction': 'why_verify_identity', 'candidates': ['why_verify_identity', 'country_support', 'get_physical_card', 'card_payment_not_recognised'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "What should I do if I see a payment I didn't make?", 'target': 'card_payment_not_recognised', 'prediction': 'get_physical_card', 'candidates': ['top_up_by_card_charge', 'card_payment_not_recognised', 'country_support', 'get_physical_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'I found an unauthorized card payment', 'target': 'card_payment_not_recognised', 'prediction': 'activate_my_card', 'candidates': ['wrong_exchange_rate_for_cash_withdrawal', 'activate_my_card', 'top_up_by_card_charge', 'card_payment_not_recognised'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'there is a transaction on my account that i didnt make', 'target': 'card_payment_not_recognised', 'prediction': 'activate_my_card', 'candidates': ['country_support', 'card_payment_not_recognised', 'top_up_by_card_charge', 'activate_my_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "I don't recall making that payment?", 'target': 'card_payment_not_recognised', 'prediction': 'why_verify_identity', 'candidates': ['country_support', 'change_pin', 'card_payment_not_recognised', 'why_verify_identity'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "There are some payments with my card that I don't know where they are coming from. Just checked today and it's been since a couple days as well! What to I do? Can I revert this retrospectively? My card needs to be frozen immediately as well!", 'target': 'card_payment_not_recognised', 'prediction': 'verify_my_identity', 'candidates': ['contactless_not_working', 'card_payment_not_recognised', 'verify_my_identity', 'change_pin'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'Has someone accessed my card there are payments I did not make that are showing up on the app.', 'target': 'card_payment_not_recognised', 'prediction': 'verify_my_identity', 'candidates': ['verify_my_identity', 'why_verify_identity', 'card_payment_not_recognised', 'top_up_failed'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "There are some strange charges on my card that I didn't make from a few days ago.  Is my card stolen and if so should I cancel it?  Will I get refunded the charges?", 'target': 'card_payment_not_recognised', 'prediction': 'change_pin', 'candidates': ['card_payment_not_recognised', 'change_pin', 'top_up_by_card_charge', 'country_support'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "Why would there be a payment on my account I don't recognize?", 'target': 'card_payment_not_recognised', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'card_payment_not_recognised', 'verify_my_identity', 'get_physical_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "I have a payment I didn't do", 'target': 'card_payment_not_recognised', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'why_verify_identity', 'card_payment_not_recognised'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'There is a payment that is not mine.', 'target': 'card_payment_not_recognised', 'prediction': 'verify_my_identity', 'candidates': ['verify_my_identity', 'why_verify_identity', 'card_payment_not_recognised', 'activate_my_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "Can you freeze my account?  I just saw there are transactions on my account that I don't recognize.  How can I fix this?", 'target': 'card_payment_not_recognised', 'prediction': 'pending_card_payment', 'candidates': ['wrong_exchange_rate_for_cash_withdrawal', 'change_pin', 'pending_card_payment', 'card_payment_not_recognised'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'There is a card payment I do not recognise.', 'target': 'card_payment_not_recognised', 'prediction': 'activate_my_card', 'candidates': ['country_support', 'why_verify_identity', 'activate_my_card', 'card_payment_not_recognised'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "I don't recognise a card payment", 'target': 'card_payment_not_recognised', 'prediction': 'activate_my_card', 'candidates': ['change_pin', 'card_payment_not_recognised', 'activate_my_card', 'top_up_by_card_charge'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'Payment i did not do', 'target': 'card_payment_not_recognised', 'prediction': 'get_physical_card', 'candidates': ['get_physical_card', 'pending_card_payment', 'card_payment_not_recognised', 'change_pin'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "Please put a freeze on my card,  I am worried there has been some payments on it and I don't know what for.", 'target': 'card_payment_not_recognised', 'prediction': 'activate_my_card', 'candidates': ['card_payment_not_recognised', 'activate_my_card', 'verify_my_identity', 'change_pin'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'I think my statement has a fraudulent charge', 'target': 'card_payment_not_recognised', 'prediction': 'pending_card_payment', 'candidates': ['country_support', 'card_payment_not_recognised', 'pending_card_payment', 'top_up_failed'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "I'm concerned about a payment that has recently been made on my card.  I do not recognize the name that the transaction went to, can you help me?", 'target': 'card_payment_not_recognised', 'prediction': 'verify_my_identity', 'candidates': ['change_pin', 'top_up_by_card_charge', 'card_payment_not_recognised', 'verify_my_identity'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "What should I do to get transactions off of my account if I didn't make them?  My card must have been compromised and I need to freeze it asap!", 'target': 'card_payment_not_recognised', 'prediction': 'get_physical_card', 'candidates': ['get_physical_card', 'card_payment_not_recognised', 'contactless_not_working', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'I have a payment listed in error.', 'target': 'card_payment_not_recognised', 'prediction': 'get_physical_card', 'candidates': ['get_physical_card', 'card_payment_not_recognised', 'wrong_exchange_rate_for_cash_withdrawal', 'contactless_not_working'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'Are contactless payments enabled on my new card?', 'target': 'contactless_not_working', 'prediction': 'activate_my_card', 'candidates': ['why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal', 'activate_my_card', 'contactless_not_working'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'My contanctless has stopped working', 'target': 'contactless_not_working', 'prediction': 'why_verify_identity', 'candidates': ['contactless_not_working', 'get_physical_card', 'why_verify_identity', 'activate_my_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'what is required documents for new card process ?', 'target': 'contactless_not_working', 'prediction': 'verify_my_identity', 'candidates': ['get_physical_card', 'contactless_not_working', 'top_up_failed', 'verify_my_identity'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "I don't know what's wrong, my contactless stopped working. Tried it in a few different places today and it didn't work in any of them.", 'target': 'contactless_not_working', 'prediction': 'wrong_exchange_rate_for_cash_withdrawal', 'candidates': ['top_up_by_card_charge', 'wrong_exchange_rate_for_cash_withdrawal', 'card_payment_not_recognised', 'contactless_not_working'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "I've been unable to pay contactless with my card for over a week now. Is there a way to fix this issue?", 'target': 'contactless_not_working', 'prediction': 'activate_my_card', 'candidates': ['activate_my_card', 'contactless_not_working', 'change_pin', 'country_support'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'My contactless payments have not been completed successfully.', 'target': 'contactless_not_working', 'prediction': 'verify_my_identity', 'candidates': ['top_up_failed', 'contactless_not_working', 'get_physical_card', 'verify_my_identity'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "Do I need to replace my card if it didn't work when I tried to pay contactless at the shop today?", 'target': 'contactless_not_working', 'prediction': 'why_verify_identity', 'candidates': ['wrong_exchange_rate_for_cash_withdrawal', 'why_verify_identity', 'contactless_not_working', 'top_up_by_card_charge'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "For some reason my contactless has stopped working. I don't know what the problem is. What can I do?", 'target': 'contactless_not_working', 'prediction': 'verify_my_identity', 'candidates': ['why_verify_identity', 'contactless_not_working', 'wrong_exchange_rate_for_cash_withdrawal', 'verify_my_identity'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "I couldn't use my contactless this morning, it wasn't accepted.", 'target': 'contactless_not_working', 'prediction': 'why_verify_identity', 'candidates': ['why_verify_identity', 'change_pin', 'card_payment_not_recognised', 'contactless_not_working'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "Contactless isn't working for me", 'target': 'contactless_not_working', 'prediction': 'why_verify_identity', 'candidates': ['contactless_not_working', 'activate_my_card', 'top_up_failed', 'why_verify_identity'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'Fix my contactless', 'target': 'contactless_not_working', 'prediction': 'activate_my_card', 'candidates': ['pending_card_payment', 'top_up_failed', 'activate_my_card', 'contactless_not_working'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'My contactless is not working', 'target': 'contactless_not_working', 'prediction': 'why_verify_identity', 'candidates': ['change_pin', 'top_up_by_card_charge', 'why_verify_identity', 'contactless_not_working'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "For some reason my contactless won't work for me.", 'target': 'contactless_not_working', 'prediction': 'activate_my_card', 'candidates': ['wrong_exchange_rate_for_cash_withdrawal', 'activate_my_card', 'contactless_not_working', 'card_payment_not_recognised'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "I'm unable to use contactless payments for purchases.", 'target': 'contactless_not_working', 'prediction': 'activate_my_card', 'candidates': ['card_payment_not_recognised', 'pending_card_payment', 'activate_my_card', 'contactless_not_working'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "When I tried to pay for my bus ride, the contactless payment wouldn't work. How can I prevent this in the future?", 'target': 'contactless_not_working', 'prediction': 'pending_card_payment', 'candidates': ['country_support', 'contactless_not_working', 'card_payment_not_recognised', 'pending_card_payment'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "The NFC payment wouldn't work on the bus today. Help?", 'target': 'contactless_not_working', 'prediction': 'activate_my_card', 'candidates': ['get_physical_card', 'top_up_failed', 'contactless_not_working', 'activate_my_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'I live in the US but want to get a card', 'target': 'country_support', 'prediction': 'activate_my_card', 'candidates': ['get_physical_card', 'wrong_exchange_rate_for_cash_withdrawal', 'country_support', 'activate_my_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'Are your cards available in Europe?', 'target': 'country_support', 'prediction': 'pending_card_payment', 'candidates': ['top_up_by_card_charge', 'country_support', 'pending_card_payment', 'change_pin'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'Can I use your app if I am from the EU?', 'target': 'country_support', 'prediction': 'activate_my_card', 'candidates': ['activate_my_card', 'top_up_failed', 'country_support', 'pending_card_payment'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'What countries are supported?', 'target': 'country_support', 'prediction': 'why_verify_identity', 'candidates': ['pending_card_payment', 'top_up_failed', 'why_verify_identity', 'country_support'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'i live in the US can i still get a card?', 'target': 'country_support', 'prediction': 'why_verify_identity', 'candidates': ['card_payment_not_recognised', 'why_verify_identity', 'get_physical_card', 'country_support'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'Can I get a card outside the UK?', 'target': 'country_support', 'prediction': 'top_up_by_card_charge', 'candidates': ['country_support', 'top_up_failed', 'top_up_by_card_charge', 'change_pin'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "I'm not in the UK, can I still get a card?", 'target': 'country_support', 'prediction': 'pending_card_payment', 'candidates': ['card_payment_not_recognised', 'pending_card_payment', 'country_support', 'change_pin'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "I'm not in the UK, is it possible for me to still get a card?", 'target': 'country_support', 'prediction': 'pending_card_payment', 'candidates': ['country_support', 'contactless_not_working', 'pending_card_payment', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'Now that I live in the US how can I get a card?', 'target': 'country_support', 'prediction': 'pending_card_payment', 'candidates': ['pending_card_payment', 'country_support', 'top_up_by_card_charge', 'top_up_failed'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'Can I get one of your cards in the EU', 'target': 'country_support', 'prediction': 'activate_my_card', 'candidates': ['verify_my_identity', 'card_payment_not_recognised', 'activate_my_card', 'country_support'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "Can I get a card despite the fact I'm not in the UK?", 'target': 'country_support', 'prediction': 'verify_my_identity', 'candidates': ['why_verify_identity', 'country_support', 'verify_my_identity', 'pending_card_payment'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'Where do I need to live to get support?', 'target': 'country_support', 'prediction': 'verify_my_identity', 'candidates': ['verify_my_identity', 'contactless_not_working', 'country_support', 'get_physical_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'Do you support the EU?', 'target': 'country_support', 'prediction': 'activate_my_card', 'candidates': ['card_payment_not_recognised', 'pending_card_payment', 'country_support', 'activate_my_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'how do i get a card if i am in the usa', 'target': 'country_support', 'prediction': 'verify_my_identity', 'candidates': ['top_up_by_card_charge', 'card_payment_not_recognised', 'verify_my_identity', 'country_support'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'Please help me get a new card, I reside in the United States.', 'target': 'country_support', 'prediction': 'verify_my_identity', 'candidates': ['country_support', 'pending_card_payment', 'verify_my_identity', 'why_verify_identity'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "Is it possible to get a card if I'm not in the UK?", 'target': 'country_support', 'prediction': 'why_verify_identity', 'candidates': ['country_support', 'top_up_by_card_charge', 'change_pin', 'why_verify_identity'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'From what countries can I use this product?', 'target': 'country_support', 'prediction': 'why_verify_identity', 'candidates': ['card_payment_not_recognised', 'country_support', 'why_verify_identity', 'get_physical_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'Where can I find your locations?', 'target': 'country_support', 'prediction': 'change_pin', 'candidates': ['country_support', 'contactless_not_working', 'top_up_failed', 'change_pin'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'Can I get a card even if I live outside the UK?', 'target': 'country_support', 'prediction': 'activate_my_card', 'candidates': ['country_support', 'why_verify_identity', 'activate_my_card', 'verify_my_identity'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'I am from the EU, can I sign up?', 'target': 'country_support', 'prediction': 'verify_my_identity', 'candidates': ['verify_my_identity', 'country_support', 'change_pin', 'why_verify_identity'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'I would like to receive and use the card in Europe, is that possible?', 'target': 'country_support', 'prediction': 'verify_my_identity', 'candidates': ['verify_my_identity', 'pending_card_payment', 'country_support', 'contactless_not_working'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'What countries do you support?', 'target': 'country_support', 'prediction': 'verify_my_identity', 'candidates': ['contactless_not_working', 'country_support', 'top_up_by_card_charge', 'verify_my_identity'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'What countries are you currently in?', 'target': 'country_support', 'prediction': 'pending_card_payment', 'candidates': ['country_support', 'wrong_exchange_rate_for_cash_withdrawal', 'top_up_failed', 'pending_card_payment'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'What locations are you in?', 'target': 'country_support', 'prediction': 'top_up_by_card_charge', 'candidates': ['top_up_by_card_charge', 'contactless_not_working', 'country_support', 'top_up_failed'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'What do I do with my card PIN?', 'target': 'get_physical_card', 'prediction': 'pending_card_payment', 'candidates': ['country_support', 'pending_card_payment', 'contactless_not_working', 'get_physical_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'Is the PIN delivered separately?', 'target': 'get_physical_card', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'verify_my_identity', 'get_physical_card', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'I do not have my pin', 'target': 'get_physical_card', 'prediction': 'why_verify_identity', 'candidates': ['change_pin', 'get_physical_card', 'why_verify_identity', 'activate_my_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'How do I set-up my PIN for the new card?', 'target': 'get_physical_card', 'prediction': 'change_pin', 'candidates': ['pending_card_payment', 'get_physical_card', 'why_verify_identity', 'change_pin'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'How do I find my card PIN?', 'target': 'get_physical_card', 'prediction': 'change_pin', 'candidates': ['top_up_by_card_charge', 'get_physical_card', 'change_pin', 'verify_my_identity'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "I don't know where to look to find my PIN.", 'target': 'get_physical_card', 'prediction': 'verify_my_identity', 'candidates': ['verify_my_identity', 'get_physical_card', 'top_up_failed', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'where in the app can i find out about my new pin', 'target': 'get_physical_card', 'prediction': 'verify_my_identity', 'candidates': ['pending_card_payment', 'verify_my_identity', 'get_physical_card', 'why_verify_identity'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'Where in the app can I find my PIN?', 'target': 'get_physical_card', 'prediction': 'verify_my_identity', 'candidates': ['verify_my_identity', 'get_physical_card', 'wrong_exchange_rate_for_cash_withdrawal', 'top_up_failed'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'I just got my new card but am not sure how to check its PIN.', 'target': 'get_physical_card', 'prediction': 'activate_my_card', 'candidates': ['get_physical_card', 'top_up_failed', 'contactless_not_working', 'activate_my_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'do i need a pin', 'target': 'get_physical_card', 'prediction': 'activate_my_card', 'candidates': ['country_support', 'get_physical_card', 'activate_my_card', 'contactless_not_working'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'If I need a PIN for my card, where is it located?', 'target': 'get_physical_card', 'prediction': 'why_verify_identity', 'candidates': ['pending_card_payment', 'country_support', 'get_physical_card', 'why_verify_identity'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'what is my card PIN', 'target': 'get_physical_card', 'prediction': 'verify_my_identity', 'candidates': ['get_physical_card', 'change_pin', 'verify_my_identity', 'activate_my_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "And what about the Card's PIN?", 'target': 'get_physical_card', 'prediction': 'activate_my_card', 'candidates': ['contactless_not_working', 'country_support', 'activate_my_card', 'get_physical_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'About this card PIN?', 'target': 'get_physical_card', 'prediction': 'verify_my_identity', 'candidates': ['verify_my_identity', 'card_payment_not_recognised', 'why_verify_identity', 'get_physical_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'I received my card but not my PIN.', 'target': 'get_physical_card', 'prediction': 'activate_my_card', 'candidates': ['change_pin', 'get_physical_card', 'contactless_not_working', 'activate_my_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'what is the process for setting up a pin', 'target': 'get_physical_card', 'prediction': 'verify_my_identity', 'candidates': ['verify_my_identity', 'country_support', 'pending_card_payment', 'get_physical_card'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': 'Does my PIN come with my card?', 'target': 'get_physical_card', 'prediction': 'verify_my_identity', 'candidates': ['wrong_exchange_rate_for_cash_withdrawal', 'verify_my_identity', 'get_physical_card', 'top_up_by_card_charge'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'nanojev', 'panel': 'dynamic', 'message': "Why isn't my PIN available yet?", 'target': 'get_physical_card', 'prediction': 'pending_card_payment', 'candidates': ['get_physical_card', 'top_up_by_card_charge', 'pending_card_payment', 'card_payment_not_recognised'], 'schema_valid': True, 'generated_text': ''}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': "Why can't I activate my card?", 'target': 'activate_my_card', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "why_verify_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'WHAT CAN I DO AFTER THE CARD MISSING', 'target': 'activate_my_card', 'prediction': 'pending_card_payment', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "pending_card_payment"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': "I tried activating my bill of fare and it didn't workplace", 'target': 'activate_my_card', 'prediction': 'top_up_failed', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "top_up_failed"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'Any hidden fees associated with the international card if money is added?', 'target': 'top_up_by_card_charge', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "why_verify_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'How can I check to see if my top-up worked?', 'target': 'top_up_failed', 'prediction': 'top_up_by_card_charge', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "top_up_by_card_charge"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'Why did my top up not work?', 'target': 'top_up_failed', 'prediction': 'top_up_by_card_charge', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "top_up_by_card_charge"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'I need to know why my credit card was declined for top up? What is the deal here and why did it not go through?', 'target': 'top_up_failed', 'prediction': 'top_up_by_card_charge', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "top_up_by_card_charge"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'Can you tell me if my pop-up went through?', 'target': 'top_up_failed', 'prediction': 'top_up_by_card_charge', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "top_up_by_card_charge"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': "can you check why I wasn't able to top up?", 'target': 'top_up_failed', 'prediction': 'top_up_by_card_charge', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "top_up_by_card_charge"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'do I need to verify my identity every time I log into my account?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'Why am I required to verify my identity?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'Can I make transfers before identity verification?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'What is the need to verify my identity?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'Do I have to do the identity check?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'Can I use my account without verifying my identity?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'Why is identity verification mandatory?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'Do I have to do an identity check?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'I would like to not have to do the identity verification.', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'Can I still use my account, even though the identity verification has not passed yet?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'Why do you need so much information to verify my identity?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'What is the purpose for the identity check', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': "I don't like that I have to fill out so much information about my identity.", 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'What is the point of an identity check?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'How come you need to know my identity?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'May I use my account now, even though the identity verification has not gone through yet?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'Do I need to verify my identity to use my account?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'Why am I required to do an identity check?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'Why am I being asked to have an identity check', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'Can i make transactions before identity verification is complete?', 'target': 'why_verify_identity', 'prediction': 'pending_card_payment', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "pending_card_payment"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'Do I really need to verify my identity?', 'target': 'why_verify_identity', 'prediction': 'verify_my_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': "I made a withdraw from the ATM this past holiday and it seems like a was charged too much. If I would have   known about these charges I wouldn't have made a withdraw.", 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'top_up_by_card_charge', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "top_up_by_card_charge"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'It seems I was overcharged when I used an ATM while on vacation. If I knew about your fees in advance I sure would have gone somewhere else.', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'top_up_failed', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "top_up_failed"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'I feel like too much money was taken during my currency exchange.', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'why_verify_identity', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "why_verify_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'fixed', 'message': 'I need the right foreign money, at the right rate, at ATMs, when I go abroad.', 'target': 'wrong_exchange_rate_for_cash_withdrawal', 'prediction': 'top_up_by_card_charge', 'candidates': ['activate_my_card', 'change_pin', 'pending_card_payment', 'top_up_by_card_charge', 'top_up_failed', 'verify_my_identity', 'why_verify_identity', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "top_up_by_card_charge"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'I have a strange payment in my statement', 'target': 'card_payment_not_recognised', 'prediction': None, 'candidates': ['why_verify_identity', 'card_payment_not_recognised', 'top_up_failed', 'pending_card_payment'], 'schema_valid': False, 'generated_text': '```json\n{"route": "pending_card_payment"}\n```'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'I found an unauthorized card payment', 'target': 'card_payment_not_recognised', 'prediction': None, 'candidates': ['wrong_exchange_rate_for_cash_withdrawal', 'activate_my_card', 'top_up_by_card_charge', 'card_payment_not_recognised'], 'schema_valid': False, 'generated_text': '```json\n{"route": "card_payment_not_recognised"}\n```'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'there is a transaction on my account that i didnt make', 'target': 'card_payment_not_recognised', 'prediction': None, 'candidates': ['country_support', 'card_payment_not_recognised', 'top_up_by_card_charge', 'activate_my_card'], 'schema_valid': False, 'generated_text': '```json\n{"route": "country_support"}\n```'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': "There are some payments with my card that I don't know where they are coming from. Just checked today and it's been since a couple days as well! What to I do? Can I revert this retrospectively? My card needs to be frozen immediately as well!", 'target': 'card_payment_not_recognised', 'prediction': 'change_pin', 'candidates': ['contactless_not_working', 'card_payment_not_recognised', 'verify_my_identity', 'change_pin'], 'schema_valid': True, 'generated_text': '{"route": "change_pin"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': "There are some strange charges on my card that I didn't make from a few days ago.  Is my card stolen and if so should I cancel it?  Will I get refunded the charges?", 'target': 'card_payment_not_recognised', 'prediction': 'country_support', 'candidates': ['card_payment_not_recognised', 'change_pin', 'top_up_by_card_charge', 'country_support'], 'schema_valid': True, 'generated_text': '{"route": "country_support"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': "I don't recognise a card payment", 'target': 'card_payment_not_recognised', 'prediction': None, 'candidates': ['change_pin', 'card_payment_not_recognised', 'activate_my_card', 'top_up_by_card_charge'], 'schema_valid': False, 'generated_text': '```json\n{"route": "card_payment_not_recognised"}\n```'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'Payment i did not do', 'target': 'card_payment_not_recognised', 'prediction': None, 'candidates': ['get_physical_card', 'pending_card_payment', 'card_payment_not_recognised', 'change_pin'], 'schema_valid': False, 'generated_text': '```json\n{"route": "card_payment_not_recognised"}\n```'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'I think my statement has a fraudulent charge', 'target': 'card_payment_not_recognised', 'prediction': None, 'candidates': ['country_support', 'card_payment_not_recognised', 'pending_card_payment', 'top_up_failed'], 'schema_valid': False, 'generated_text': '```json\n{"route": "country_support"}\n```'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': "What should I do to get transactions off of my account if I didn't make them?  My card must have been compromised and I need to freeze it asap!", 'target': 'card_payment_not_recognised', 'prediction': 'contactless_not_working', 'candidates': ['get_physical_card', 'card_payment_not_recognised', 'contactless_not_working', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "contactless_not_working"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'what is required documents for new card process ?', 'target': 'contactless_not_working', 'prediction': 'verify_my_identity', 'candidates': ['get_physical_card', 'contactless_not_working', 'top_up_failed', 'verify_my_identity'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'Can you help my fix my contactless?', 'target': 'contactless_not_working', 'prediction': 'country_support', 'candidates': ['contactless_not_working', 'top_up_failed', 'pending_card_payment', 'country_support'], 'schema_valid': True, 'generated_text': '{"route": "country_support"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': "I'm not in the UK, can I still get a card?", 'target': 'country_support', 'prediction': None, 'candidates': ['card_payment_not_recognised', 'pending_card_payment', 'country_support', 'change_pin'], 'schema_valid': False, 'generated_text': '```json\n{"route": "country_support"}\n```'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': "Can I get a card despite the fact I'm not in the UK?", 'target': 'country_support', 'prediction': 'verify_my_identity', 'candidates': ['why_verify_identity', 'country_support', 'verify_my_identity', 'pending_card_payment'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'how do i get a card if i am in the usa', 'target': 'country_support', 'prediction': None, 'candidates': ['top_up_by_card_charge', 'card_payment_not_recognised', 'verify_my_identity', 'country_support'], 'schema_valid': False, 'generated_text': '```json\n{"route": "country_support"}\n```'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'Can I get a card even if I live outside the UK?', 'target': 'country_support', 'prediction': 'verify_my_identity', 'candidates': ['country_support', 'why_verify_identity', 'activate_my_card', 'verify_my_identity'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'I am from the EU, can I sign up?', 'target': 'country_support', 'prediction': 'verify_my_identity', 'candidates': ['verify_my_identity', 'country_support', 'change_pin', 'why_verify_identity'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'I do not have my pin', 'target': 'get_physical_card', 'prediction': 'why_verify_identity', 'candidates': ['change_pin', 'get_physical_card', 'why_verify_identity', 'activate_my_card'], 'schema_valid': True, 'generated_text': '{"route": "why_verify_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'Are PIN separately?', 'target': 'get_physical_card', 'prediction': 'change_pin', 'candidates': ['get_physical_card', 'country_support', 'change_pin', 'pending_card_payment'], 'schema_valid': True, 'generated_text': '{"route": "change_pin"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'How do I set-up my PIN for the new card?', 'target': 'get_physical_card', 'prediction': 'change_pin', 'candidates': ['pending_card_payment', 'get_physical_card', 'why_verify_identity', 'change_pin'], 'schema_valid': True, 'generated_text': '{"route": "change_pin"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'How do I find my card PIN?', 'target': 'get_physical_card', 'prediction': 'change_pin', 'candidates': ['top_up_by_card_charge', 'get_physical_card', 'change_pin', 'verify_my_identity'], 'schema_valid': True, 'generated_text': '{"route": "change_pin"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'is my card PIN saved in the app', 'target': 'get_physical_card', 'prediction': 'top_up_by_card_charge', 'candidates': ['get_physical_card', 'country_support', 'top_up_failed', 'top_up_by_card_charge'], 'schema_valid': True, 'generated_text': '{"route": "top_up_by_card_charge"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': "I don't know where to look to find my PIN.", 'target': 'get_physical_card', 'prediction': 'verify_my_identity', 'candidates': ['verify_my_identity', 'get_physical_card', 'top_up_failed', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'Where in the app can I find my PIN?', 'target': 'get_physical_card', 'prediction': 'verify_my_identity', 'candidates': ['verify_my_identity', 'get_physical_card', 'wrong_exchange_rate_for_cash_withdrawal', 'top_up_failed'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'I just got my new card but am not sure how to check its PIN.', 'target': 'get_physical_card', 'prediction': 'activate_my_card', 'candidates': ['get_physical_card', 'top_up_failed', 'contactless_not_working', 'activate_my_card'], 'schema_valid': True, 'generated_text': '{"route": "activate_my_card"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'do i need a pin', 'target': 'get_physical_card', 'prediction': 'contactless_not_working', 'candidates': ['country_support', 'get_physical_card', 'activate_my_card', 'contactless_not_working'], 'schema_valid': True, 'generated_text': '{"route": "contactless_not_working"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'If I need a PIN for my card, where is it located?', 'target': 'get_physical_card', 'prediction': 'why_verify_identity', 'candidates': ['pending_card_payment', 'country_support', 'get_physical_card', 'why_verify_identity'], 'schema_valid': True, 'generated_text': '{"route": "why_verify_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'what is my card PIN', 'target': 'get_physical_card', 'prediction': 'change_pin', 'candidates': ['get_physical_card', 'change_pin', 'verify_my_identity', 'activate_my_card'], 'schema_valid': True, 'generated_text': '{"route": "change_pin"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': "And what about the Card's PIN?", 'target': 'get_physical_card', 'prediction': 'activate_my_card', 'candidates': ['contactless_not_working', 'country_support', 'activate_my_card', 'get_physical_card'], 'schema_valid': True, 'generated_text': '{"route": "activate_my_card"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'About this card PIN?', 'target': 'get_physical_card', 'prediction': None, 'candidates': ['verify_my_identity', 'card_payment_not_recognised', 'why_verify_identity', 'get_physical_card'], 'schema_valid': False, 'generated_text': '```json\n{"route": "why_verify_identity"}\n```'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'I received my card but not my PIN.', 'target': 'get_physical_card', 'prediction': 'contactless_not_working', 'candidates': ['change_pin', 'get_physical_card', 'contactless_not_working', 'activate_my_card'], 'schema_valid': True, 'generated_text': '{"route": "contactless_not_working"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': "Why can't I see my PIN?", 'target': 'get_physical_card', 'prediction': 'contactless_not_working', 'candidates': ['contactless_not_working', 'wrong_exchange_rate_for_cash_withdrawal', 'pending_card_payment', 'get_physical_card'], 'schema_valid': True, 'generated_text': '{"route": "contactless_not_working"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'Does my PIN come with my card?', 'target': 'get_physical_card', 'prediction': 'verify_my_identity', 'candidates': ['wrong_exchange_rate_for_cash_withdrawal', 'verify_my_identity', 'get_physical_card', 'top_up_by_card_charge'], 'schema_valid': True, 'generated_text': '{"route": "verify_my_identity"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': "Why isn't my PIN available yet?", 'target': 'get_physical_card', 'prediction': 'pending_card_payment', 'candidates': ['get_physical_card', 'top_up_by_card_charge', 'pending_card_payment', 'card_payment_not_recognised'], 'schema_valid': True, 'generated_text': '{"route": "pending_card_payment"}'}, {'project_id': 'qwen_generative', 'panel': 'dynamic', 'message': 'Please help in finding my card PIN, thank you!', 'target': 'get_physical_card', 'prediction': 'wrong_exchange_rate_for_cash_withdrawal', 'candidates': ['get_physical_card', 'card_payment_not_recognised', 'top_up_by_card_charge', 'wrong_exchange_rate_for_cash_withdrawal'], 'schema_valid': True, 'generated_text': '{"route": "wrong_exchange_rate_for_cash_withdrawal"}'}]
rows = EMBEDDED_ROWS
import csv, io
url_base = 'https://raw.githubusercontent.com/LambertLeong/LambertLeong.github.io/master/assets/notebooks/open-decision-models/results/named-project-gpu-2026-10-09/'
try:
    downloaded = []
    for panel in ['fixed', 'dynamic']:
        with urllib.request.urlopen(url_base + 'open-jev-project-' + panel + '.csv', timeout=10) as response:
            table = csv.DictReader(io.StringIO(response.read().decode('utf-8')))
            downloaded.extend(dict(project_id=r['project_id'], configuration=r['configuration'], panel=panel,
                                   accuracy=float(r['accuracy']), macro_f1=float(r['macro_f1'])) for r in table)
    if len(downloaded) != 16 or {r['project_id'] for r in downloaded} != {r['project_id'] for r in EMBEDDED_ROWS}:
        raise ValueError('Public comparison is incomplete; use the embedded accepted copy.')
    rows = downloaded
    print('Loaded public accepted comparison.')
except (OSError, ValueError, KeyError) as exc:
    print('Using embedded accepted results:', type(exc).__name__)
frame = pd.DataFrame(rows)
def show_panel(panel):
    subset = frame[frame.panel == panel].sort_values('accuracy')
    fig = px.bar(subset, x='accuracy', y='configuration', orientation='h', range_x=[0,1],
                 title=f'BANKING77: {panel} candidates', text=subset.accuracy.map(lambda x: f'{x:.1%}'))
    fig.update_xaxes(tickformat='.0%')
    fig.show()
widgets.interact(show_panel, panel=['fixed', 'dynamic'])

In [ ]:
def inspect_errors(project, panel):
    display(pd.DataFrame([r for r in EMBEDDED_ERRORS if r['panel'] == panel and r['project_id'] == project]))
print('Actual errors from repeat 1: NanoJev or Qwen generation.')
widgets.interact(inspect_errors, project=['nanojev', 'qwen_generative'], panel=['fixed', 'dynamic'])

## Same weights: JSON generation vs one forward pass

This small experiment times two request paths on the device you selected. It is a playground, not the archived benchmark. Both paths see the same message and candidate descriptions, with instructions suited to their outputs. The direct path scores single-token option labels; it is not a trained native decision model.

We warm both paths, synchronize CUDA, and include tokenization and output handling in the timer. A generated response is validated once, with no repairs or retry. The table reports the actual output token count, including EOS if emitted.

In [ ]:
# Release AnyJev before loading the comparison model, to fit small runtimes.
import gc, time
if 'model' in globals(): del model
if 'backend' in globals(): del backend
gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, revision=REVISION)
qwen = AutoModelForCausalLM.from_pretrained(CHECKPOINT, revision=REVISION, torch_dtype=torch.float32).to(DEVICE).eval()
labels = [chr(65+i) for i in range(len(candidates))]
label_ids = [tokenizer.encode(label, add_special_tokens=False) for label in labels]
assert all(len(ids)==1 for ids in label_ids), 'Each option label must be a single token'
menu = '\n'.join(f'{label}: {candidate}' for label, candidate in zip(labels, candidates))
base = f'Route this banking request: {query}\nCandidates:\n{menu}\n'
def parse_generated_route(text, allowed):
    def unique_object(pairs):
        result = {}
        for key, value in pairs:
            if key in result: raise ValueError('Duplicate JSON key')
            result[key] = value
        return result
    try:
        value = json.loads(text, object_pairs_hook=unique_object)
        valid = (isinstance(value, dict) and set(value) == {'route'}
                 and isinstance(value['route'], str) and value['route'] in allowed)
    except (ValueError, TypeError): valid = False
    return (value['route'], True) if valid else (None, False)
def request(method):
    if DEVICE == 'cuda': torch.cuda.synchronize()
    started = time.perf_counter()
    instruction = ('Reply with JSON ONLY: {"route": "<candidate text>"}' if method == 'generation'
                   else 'Reply with the single letter of the best candidate.')
    text = tokenizer.apply_chat_template([{'role':'user','content':base+instruction}], tokenize=False,
                                         add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors='pt').to(DEVICE)
    with torch.inference_mode():
        if method == 'generation':
            output = qwen.generate(**inputs, do_sample=False, max_new_tokens=40, pad_token_id=tokenizer.eos_token_id)
            completion = output[0, inputs['input_ids'].shape[1]:]
            text_out = tokenizer.decode(completion, skip_special_tokens=True)
            tokens = int(completion.numel())
            route, valid = parse_generated_route(text_out, candidates)
        else:
            logits = qwen(**inputs).logits[0,-1,[ids[0] for ids in label_ids]]
            probabilities = logits.softmax(-1).tolist()
            route = candidates[max(range(len(candidates)), key=lambda i:probabilities[i])]
            tokens, valid = 0, True
    if DEVICE == 'cuda': torch.cuda.synchronize()
    return dict(method=method, route=route, schema_valid=valid, output_tokens=tokens,
                forward_passes=tokens if method=='generation' else 1, latency_ms=(time.perf_counter()-started)*1000)
for method in ['generation','readout']: request(method)
comparison = pd.DataFrame([request(method) for method in ['generation','readout']])
display(comparison)
print('Observed latency ratio (generation/readout):', round(comparison.latency_ms.iloc[0]/comparison.latency_ms.iloc[1],2))

## Cost is a rate assumption multiplied by measured time

Local GPU estimate per 1,000 sequential requests = mean warm seconds × 1,000 / 3,600 × GPU dollars/hour. This excludes model loading, idle time, batching and service overhead. It is not hosted API pricing. The rate below is an editable scenario, not a quoted provider price.

In [ ]:
GPU_USD_PER_HOUR = 2.50
if DEVICE == 'cuda':
    display(comparison.assign(usd_per_1000_requests=comparison.latency_ms/1000*1000/3600*GPU_USD_PER_HOUR))
else:
    print('CPU runtime: no GPU dollar estimate is attached to these timings.')

## Reproduce the full benchmark

Everything above explores the saved results. This section re-runs the experiments from the article: all eight configurations (Kev, NanoJev, Laya, Verdict, AnyJev, Simple Jev, OpenJev and the Qwen3-0.6B JSON-generation baseline) on 200 fixed-route and 100 changing-candidate BANKING77 requests, repeated twice. Each project is installed from its pinned official commit and checkpoint, smoke-tested, then measured.

**Requirements:** a Colab or local NVIDIA GPU runtime, `git`, and enough time for seven separate installs and model downloads (plan on well over an hour). CPU fallback is disabled.

Set `RUN_FULL_BENCHMARK = True` in the next cell to enable it. With the default `False`, the cells below do nothing, so Run All stays quick. When it finishes, the last cell writes the figures, a hash manifest and a results ZIP, and offers the ZIP for download in Colab. Latency depends on your GPU and session, so expect your timings to differ from the article; accuracy should match.

In [ ]:
# Set to True to re-run the full eight-configuration benchmark (GPU required, long-running).
RUN_FULL_BENCHMARK = False

In [ ]:
if RUN_FULL_BENCHMARK:
    from pathlib import Path
    import base64, hashlib, io, json, platform, sys, time, zipfile
    BUNDLE_SHA256 = '807aea8464a0d480f64c0cdc5fb61d28ddcda936b77c2005dacd70f7c242b96f'
    BUNDLE_BASE64 = '...'
    payload = base64.b64decode(BUNDLE_BASE64)
    if hashlib.sha256(payload).hexdigest() != BUNDLE_SHA256:
        raise ValueError('Benchmark bundle hash mismatch')
    ROOT = Path.cwd() / '_tmp' / ('open-jev-notebook-' + BUNDLE_SHA256[:12])
    ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for member in archive.infolist():
            target = (ROOT / member.filename).resolve()
            if not target.is_relative_to(ROOT.resolve()):
                raise ValueError('Unsafe bundle path')
        archive.extractall(ROOT)
    REGISTRY = ROOT / 'assets/notebooks/results/open-jev-project-registry.json'
    registry = json.loads(REGISTRY.read_text())
    # Reload benchmark modules so a previous notebook run cannot supply stale code.
    for module_name in list(sys.modules):
        if module_name == 'open_jev_benchmark' or module_name.startswith('open_jev_benchmark.') or module_name in {'run_open_jev_suite', 'setup_open_jev_runtime'}:
            sys.modules.pop(module_name)
    sys.path.insert(0, str(ROOT / 'scripts'))
    print('runtime:', platform.platform(), platform.python_version())
    print('sealed bundle:', BUNDLE_SHA256)
    print('registry entries:', len(registry['projects']))
    print('phase timing: runtime fingerprint complete')

In [ ]:
if RUN_FULL_BENCHMARK:
    from run_open_jev_suite import run_suite
    import subprocess, shutil, importlib.util
    DEVICE = 'cuda'
    TEST_PER_ROUTE = 25
    REPEATS = 2
    RUN_SMOKE_FIRST = True
    OUTPUT = ROOT / 'assets/notebooks/results/named-project-suite'
    WORKSPACE = ROOT / '_tmp/runtime'
    RUNS = {}
    if DEVICE == 'cuda':
        if shutil.which('git') is None:
            raise RuntimeError('Git is required for the pinned official source installs. Install git and restart this cell.')
        if shutil.which('nvidia-smi') is None:
            raise RuntimeError('No NVIDIA driver runtime detected. Select a GPU runtime before running; CPU fallback is disabled.')
        probe = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True, check=True)
        if not probe.stdout.strip():
            raise RuntimeError('No GPU detected; CPU fallback is disabled.')
        print('GPU preflight:', probe.stdout.strip())
        # nvidia-smi can see a GPU even when a process cannot create a CUDA context.
        import ctypes
        try:
            driver = ctypes.CDLL('libcuda.so.1')
            driver.cuInit.argtypes = [ctypes.c_uint]
            driver.cuInit.restype = ctypes.c_int
            driver.cuDeviceGetCount.argtypes = [ctypes.POINTER(ctypes.c_int)]
            driver.cuDeviceGetCount.restype = ctypes.c_int
            driver.cuDeviceGet.argtypes = [ctypes.POINTER(ctypes.c_int), ctypes.c_int]
            driver.cuDeviceGet.restype = ctypes.c_int
            driver.cuCtxCreate_v2.argtypes = [ctypes.POINTER(ctypes.c_void_p), ctypes.c_uint, ctypes.c_int]
            driver.cuCtxCreate_v2.restype = ctypes.c_int
            driver.cuCtxDestroy_v2.argtypes = [ctypes.c_void_p]
            driver.cuCtxDestroy_v2.restype = ctypes.c_int
            count, device_id, context = ctypes.c_int(), ctypes.c_int(), ctypes.c_void_p()
            if driver.cuInit(0) != 0 or driver.cuDeviceGetCount(ctypes.byref(count)) != 0 or count.value < 1:
                raise RuntimeError('CUDA driver cannot see an available GPU')
            if driver.cuDeviceGet(ctypes.byref(device_id), 0) != 0 or driver.cuCtxCreate_v2(ctypes.byref(context), 0, device_id) != 0:
                raise RuntimeError('CUDA context creation failed')
            if driver.cuCtxDestroy_v2(context) != 0:
                raise RuntimeError('CUDA context cleanup failed')
        except (OSError, AttributeError) as error:
            raise RuntimeError('CUDA driver library is unavailable for this notebook process') from error
        print('CUDA context preflight passed:', count.value, 'visible GPU(s)')
        ROOT.joinpath('gpu-driver-fingerprint.json').write_text(json.dumps({'nvidia_smi': probe.stdout.strip()}, indent=2))
    if importlib.util.find_spec('matplotlib') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'matplotlib==3.11.2'], check=True)
    if importlib.util.find_spec('tqdm') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'tqdm==4.67.1'], check=True)
    from tqdm.auto import tqdm
    PROGRESS_BARS = {}
    STAGE = 'setup'
    def show_progress(event):
        phase = event['phase']
        project = event['project']
        if phase == 'episode':
            key = (STAGE, project, event['repeat'], event['panel'])
            bar = PROGRESS_BARS.get(key)
            if bar is None:
                bar = tqdm(total=event['total'], desc=f"{project} {STAGE} run {event['repeat']} {event['panel']}", unit='request', leave=True)
                PROGRESS_BARS[key] = bar
            bar.update(max(0, event['done'] - bar.n))
            if event['status'] != 'runnable':
                tqdm.write(f"{project} {event['panel']} request {event['done']}: {event['status']}")
            if event['done'] == event['total']:
                bar.close()
        elif phase == 'install_cached':
            tqdm.write(f"{project}: reusing installed environment")
        elif phase in ('install', 'source', 'install_complete'):
            tqdm.write(f"{project}: {event.get('message', 'installation complete')}" +
                       (f" ({event['step']}/{event['steps']})" if 'step' in event else ''))
        elif phase in ('install_heartbeat', 'run_heartbeat'):
            tqdm.write(f"{project}: {event['elapsed_seconds']}s in {event.get('message', 'model load or inference')}; log: {event['log']}")
        elif phase == 'run_start':
            tqdm.write(f"{project} {STAGE}: starting run {event['repeat']}/{event['repeats']}; preparing data/model; log: {event['log']}")
    print('Device:', DEVICE, 'test episodes/project:', 12 * TEST_PER_ROUTE, 'repeats:', REPEATS)
    print('Each project runs a small preflight before both full repeats. Progress bars track completed requests; install and model load report every 20 seconds.')

## Kev

Official source: https://github.com/jaredpalmer/kev

Repository commit: `5e42a7a03f28134853dd3ff77461457e921e5ec1`

Checkpoint: `jaredpalmer/kev-0.8b` at `bf75a6a8848ea6960ff2ed108d9ed44c2941174f`

Backend: Kev official server

Hardware: Official server supports CPU, CUDA, or Apple Silicon; adapter merges require extra load memory

Eligibility: `runnable`. Official README publishes Kev 1.0 checkpoint revisions. Hub API resolves the full released 0.8B commit; official loader accepts repo@SHA. CPU compatibility depends on installing its Python >=3.12 and Transformers 5.17 stack. Its adapter provenance pins Qwen/Qwen3.5-0.8B-Base at dc7cdfe2ee4154fa7e30f5b51ca41bfa40174e68.

Official install: `python -m pip install "kev[serve] @ git+https://github.com/jaredpalmer/kev.git@5e42a7a03f28134853dd3ff77461457e921e5ec1"`

Local entry point: `python scripts/run_open_jev_suite.py --projects kev --device cpu --test-per-route 1`

First downloads are visible in the saved logs; no alternate checkpoint, backend, or proxy is substituted.

In [ ]:
if RUN_FULL_BENCHMARK:
    phase_started = time.perf_counter()
    if RUN_SMOKE_FIRST and DEVICE == 'cuda':
        STAGE = 'smoke'
        smoke = run_suite(OUTPUT / 'smoke' / 'kev', WORKSPACE, projects=['kev'], device=DEVICE, test_per_route=1, repeats=1, progress=show_progress)
        smoke_status = next(row for row in smoke['project_statuses'] if row['project_id'] == 'kev')
        if smoke_status['execution_status'] != 'completed':
            partial = Path(shutil.make_archive(str(ROOT / 'open-jev-project-results-partial'), 'zip', OUTPUT))
            print('Smoke test failed; return this diagnostic ZIP:', partial)
            try:
                from google.colab import files
            except ImportError:
                pass
            else:
                files.download(str(partial))
            raise RuntimeError('GPU smoke test failed for kev: ' + str(smoke_status) + '. Fix this before transferring full results.')
    STAGE = 'full'
    RUNS['kev'] = run_suite(OUTPUT / 'kev', WORKSPACE, projects=['kev'], device=DEVICE, test_per_route=TEST_PER_ROUTE, repeats=REPEATS, progress=show_progress)
    print('phase timing: kev duration_s=', round(time.perf_counter() - phase_started, 2))
    print(next(row for row in RUNS['kev']['project_statuses'] if row['project_id'] == 'kev'))

## NanoJev

Official source: https://github.com/TianyuCodings/NanoJev

Repository commit: `76fdfc9ecdca45a9bcef17991a07d3041a87685a`

Checkpoint: `C-Tianyu/NanoJev` at `047b927b30882a1138fc504821b82ac145a4b81a`

Backend: NanoJev official DecisionPredictor

Hardware: CUDA required by the official predictor; fp32, no CPU fallback

Eligibility: `runnable`. The pinned input contract supports generic text states and 2–255 textual Choice options. Its released checkpoint is trained on games, which does not make its interface game-only. The official DecisionPredictor requires CUDA. The GPU follow-up uses the exact released full checkpoint and official fp32 predictor, with native Triton overrides disabled; no retraining or game-action substitution.

Official install: `python -m pip install -r requirements-toy.txt huggingface_hub`

Local entry point: `python scripts/run_open_jev_suite.py --projects nanojev --device cuda --test-per-route 1`

First downloads are visible in the saved logs; no alternate checkpoint, backend, or proxy is substituted.

In [ ]:
if RUN_FULL_BENCHMARK:
    phase_started = time.perf_counter()
    if RUN_SMOKE_FIRST and DEVICE == 'cuda':
        STAGE = 'smoke'
        smoke = run_suite(OUTPUT / 'smoke' / 'nanojev', WORKSPACE, projects=['nanojev'], device=DEVICE, test_per_route=1, repeats=1, progress=show_progress)
        smoke_status = next(row for row in smoke['project_statuses'] if row['project_id'] == 'nanojev')
        if smoke_status['execution_status'] != 'completed':
            partial = Path(shutil.make_archive(str(ROOT / 'open-jev-project-results-partial'), 'zip', OUTPUT))
            print('Smoke test failed; return this diagnostic ZIP:', partial)
            try:
                from google.colab import files
            except ImportError:
                pass
            else:
                files.download(str(partial))
            raise RuntimeError('GPU smoke test failed for nanojev: ' + str(smoke_status) + '. Fix this before transferring full results.')
    STAGE = 'full'
    RUNS['nanojev'] = run_suite(OUTPUT / 'nanojev', WORKSPACE, projects=['nanojev'], device=DEVICE, test_per_route=TEST_PER_ROUTE, repeats=REPEATS, progress=show_progress)
    print('phase timing: nanojev duration_s=', round(time.perf_counter() - phase_started, 2))
    print(next(row for row in RUNS['nanojev']['project_statuses'] if row['project_id'] == 'nanojev'))

## Laya

Official source: https://github.com/NandhaKishorM/laya

Repository commit: `a4a8921afebfd852bba0000475cfb6ab737a124c`

Checkpoint: `convaiinnovations/laya` at `7b928d828b7b0e022f929d9bd2e44165aa270148`

Backend: Laya Python SDK

Hardware: CPU (float32) or CUDA; official eager SDK

Eligibility: `runnable`. Official SDK accepts checkpoint revision and CPU device. Hub API pins the released English checkpoint. CPU smoke inference succeeded with Laya 0.3.28, torch 2.2.2 and transformers 4.51.3; loading took 198.7 seconds on this ARM host. SDK warns about an invalid temperature for 11+ choices; these panels use at most 8.

Official install: `python -m pip install "transformers==4.51.3" "laya @ git+https://github.com/NandhaKishorM/laya.git@a4a8921afebfd852bba0000475cfb6ab737a124c"`

Local entry point: `python scripts/run_open_jev_suite.py --projects laya --device cpu --test-per-route 1`

First downloads are visible in the saved logs; no alternate checkpoint, backend, or proxy is substituted.

In [ ]:
if RUN_FULL_BENCHMARK:
    phase_started = time.perf_counter()
    if RUN_SMOKE_FIRST and DEVICE == 'cuda':
        STAGE = 'smoke'
        smoke = run_suite(OUTPUT / 'smoke' / 'laya', WORKSPACE, projects=['laya'], device=DEVICE, test_per_route=1, repeats=1, progress=show_progress)
        smoke_status = next(row for row in smoke['project_statuses'] if row['project_id'] == 'laya')
        if smoke_status['execution_status'] != 'completed':
            partial = Path(shutil.make_archive(str(ROOT / 'open-jev-project-results-partial'), 'zip', OUTPUT))
            print('Smoke test failed; return this diagnostic ZIP:', partial)
            try:
                from google.colab import files
            except ImportError:
                pass
            else:
                files.download(str(partial))
            raise RuntimeError('GPU smoke test failed for laya: ' + str(smoke_status) + '. Fix this before transferring full results.')
    STAGE = 'full'
    RUNS['laya'] = run_suite(OUTPUT / 'laya', WORKSPACE, projects=['laya'], device=DEVICE, test_per_route=TEST_PER_ROUTE, repeats=REPEATS, progress=show_progress)
    print('phase timing: laya duration_s=', round(time.perf_counter() - phase_started, 2))
    print(next(row for row in RUNS['laya']['project_statuses'] if row['project_id'] == 'laya'))

## Verdict

Official source: https://github.com/Heman10x-NGU/Verdict-open-jev

Repository commit: `30f15564821626ca5c1ad5b2638c4eb7078787dd`

Checkpoint: `heman10x/rlcd-modernbert-151m` at `8af2496eb63c7fa66d7d234e1f62629380030eb4`

Backend: Verdict DecisionEngine

Hardware: CPU float32 or CUDA via official DecisionEngine

Eligibility: `runnable`. Official ARTIFACTS.json pins every bundle file by SHA-256. The benchmark loads this exact released bundle, never DecisionEngine() default base weights. The PyTorch weight hash is verified before inference. An actual CPU attempt with Transformers 4.57.6 failed on TokenizersBackend; the runtime installer now pins 5.17.0. The optional upstream calibrator differs from the older ARTIFACTS.json hash and is not used: this benchmark applies no additional calibrator.

Official install: `python -m pip install "rlcd @ git+https://github.com/Heman10x-NGU/Verdict-open-jev.git@30f15564821626ca5c1ad5b2638c4eb7078787dd"`

Local entry point: `python scripts/run_open_jev_suite.py --projects verdict --device cpu --test-per-route 1`

First downloads are visible in the saved logs; no alternate checkpoint, backend, or proxy is substituted.

In [ ]:
if RUN_FULL_BENCHMARK:
    phase_started = time.perf_counter()
    if RUN_SMOKE_FIRST and DEVICE == 'cuda':
        STAGE = 'smoke'
        smoke = run_suite(OUTPUT / 'smoke' / 'verdict', WORKSPACE, projects=['verdict'], device=DEVICE, test_per_route=1, repeats=1, progress=show_progress)
        smoke_status = next(row for row in smoke['project_statuses'] if row['project_id'] == 'verdict')
        if smoke_status['execution_status'] != 'completed':
            partial = Path(shutil.make_archive(str(ROOT / 'open-jev-project-results-partial'), 'zip', OUTPUT))
            print('Smoke test failed; return this diagnostic ZIP:', partial)
            try:
                from google.colab import files
            except ImportError:
                pass
            else:
                files.download(str(partial))
            raise RuntimeError('GPU smoke test failed for verdict: ' + str(smoke_status) + '. Fix this before transferring full results.')
    STAGE = 'full'
    RUNS['verdict'] = run_suite(OUTPUT / 'verdict', WORKSPACE, projects=['verdict'], device=DEVICE, test_per_route=TEST_PER_ROUTE, repeats=REPEATS, progress=show_progress)
    print('phase timing: verdict duration_s=', round(time.perf_counter() - phase_started, 2))
    print(next(row for row in RUNS['verdict']['project_statuses'] if row['project_id'] == 'verdict'))

## AnyJev L0 + Qwen3-0.6B

Official source: https://github.com/nokia-applied-research/AnyJev

Repository commit: `f82fe03bc0d01c9e340b4f8880b744e78ea6116d`

Checkpoint: `Qwen/Qwen3-0.6B` at `c1899de289a04d12100db370d81485cdf75e47ca`

Backend: official AnyJev pooling/head pipeline

Hardware: CPU float32 or CUDA; same pinned 0.6B base model for adapter comparisons

Eligibility: `runnable`. Official implementation supports caller-selected causal LMs. Benchmark uses the same pinned Qwen3-0.6B base for AnyJev L0, Simple Jev baseline policy, and OpenJev HTTP server; no project-specific trained checkpoint is required.

Official install: `python -m pip install "anyjev[hf] @ git+https://github.com/nokia-applied-research/AnyJev.git@f82fe03bc0d01c9e340b4f8880b744e78ea6116d"`

Local entry point: `python scripts/run_open_jev_suite.py --projects anyjev --device cpu --test-per-route 1`

First downloads are visible in the saved logs; no alternate checkpoint, backend, or proxy is substituted.

In [ ]:
if RUN_FULL_BENCHMARK:
    phase_started = time.perf_counter()
    if RUN_SMOKE_FIRST and DEVICE == 'cuda':
        STAGE = 'smoke'
        smoke = run_suite(OUTPUT / 'smoke' / 'anyjev', WORKSPACE, projects=['anyjev'], device=DEVICE, test_per_route=1, repeats=1, progress=show_progress)
        smoke_status = next(row for row in smoke['project_statuses'] if row['project_id'] == 'anyjev')
        if smoke_status['execution_status'] != 'completed':
            partial = Path(shutil.make_archive(str(ROOT / 'open-jev-project-results-partial'), 'zip', OUTPUT))
            print('Smoke test failed; return this diagnostic ZIP:', partial)
            try:
                from google.colab import files
            except ImportError:
                pass
            else:
                files.download(str(partial))
            raise RuntimeError('GPU smoke test failed for anyjev: ' + str(smoke_status) + '. Fix this before transferring full results.')
    STAGE = 'full'
    RUNS['anyjev'] = run_suite(OUTPUT / 'anyjev', WORKSPACE, projects=['anyjev'], device=DEVICE, test_per_route=TEST_PER_ROUTE, repeats=REPEATS, progress=show_progress)
    print('phase timing: anyjev duration_s=', round(time.perf_counter() - phase_started, 2))
    print(next(row for row in RUNS['anyjev']['project_statuses'] if row['project_id'] == 'anyjev'))

## Simple Jev + Qwen3-0.6B

Official source: https://github.com/featherless-ai/simple-jev

Repository commit: `9c11582d3e631e0b118f3f3132ebbb00c62020c5`

Checkpoint: `Qwen/Qwen3-0.6B` at `c1899de289a04d12100db370d81485cdf75e47ca`

Backend: documented Simple Jev serving integration

Hardware: CPU float32 or CUDA; same pinned 0.6B base model for adapter comparisons

Eligibility: `runnable`. Official implementation supports caller-selected causal LMs. Benchmark uses the same pinned Qwen3-0.6B base for AnyJev L0, Simple Jev baseline policy, and OpenJev HTTP server; no project-specific trained checkpoint is required.

Official install: `python -m pip install "git+https://github.com/featherless-ai/simple-jev.git@9c11582d3e631e0b118f3f3132ebbb00c62020c5#subdirectory=hf-server"`

Local entry point: `python scripts/run_open_jev_suite.py --projects simple_jev --device cpu --test-per-route 1`

First downloads are visible in the saved logs; no alternate checkpoint, backend, or proxy is substituted.

In [ ]:
if RUN_FULL_BENCHMARK:
    phase_started = time.perf_counter()
    if RUN_SMOKE_FIRST and DEVICE == 'cuda':
        STAGE = 'smoke'
        smoke = run_suite(OUTPUT / 'smoke' / 'simple_jev', WORKSPACE, projects=['simple_jev'], device=DEVICE, test_per_route=1, repeats=1, progress=show_progress)
        smoke_status = next(row for row in smoke['project_statuses'] if row['project_id'] == 'simple_jev')
        if smoke_status['execution_status'] != 'completed':
            partial = Path(shutil.make_archive(str(ROOT / 'open-jev-project-results-partial'), 'zip', OUTPUT))
            print('Smoke test failed; return this diagnostic ZIP:', partial)
            try:
                from google.colab import files
            except ImportError:
                pass
            else:
                files.download(str(partial))
            raise RuntimeError('GPU smoke test failed for simple_jev: ' + str(smoke_status) + '. Fix this before transferring full results.')
    STAGE = 'full'
    RUNS['simple_jev'] = run_suite(OUTPUT / 'simple_jev', WORKSPACE, projects=['simple_jev'], device=DEVICE, test_per_route=TEST_PER_ROUTE, repeats=REPEATS, progress=show_progress)
    print('phase timing: simple_jev duration_s=', round(time.perf_counter() - phase_started, 2))
    print(next(row for row in RUNS['simple_jev']['project_statuses'] if row['project_id'] == 'simple_jev'))

## OpenJev + Qwen3-0.6B

Official source: https://github.com/lookski/openjev

Repository commit: `49eb1382c88024e0f2550005264c13847d6ec167`

Checkpoint: `Qwen/Qwen3-0.6B` at `c1899de289a04d12100db370d81485cdf75e47ca`

Backend: OpenJev official HTTP server + Qwen3-0.6B

Hardware: CPU float32 or CUDA; same pinned 0.6B base model for adapter comparisons

Eligibility: `runnable`. Official implementation supports caller-selected causal LMs. Benchmark uses the same pinned Qwen3-0.6B base for AnyJev L0, Simple Jev baseline policy, and OpenJev HTTP server; no project-specific trained checkpoint is required.

Official install: `python -m pip install "git+https://github.com/lookski/openjev.git@49eb1382c88024e0f2550005264c13847d6ec167"`

Local entry point: `python scripts/run_open_jev_suite.py --projects openjev --device cpu --test-per-route 1`

First downloads are visible in the saved logs; no alternate checkpoint, backend, or proxy is substituted.

In [ ]:
if RUN_FULL_BENCHMARK:
    phase_started = time.perf_counter()
    if RUN_SMOKE_FIRST and DEVICE == 'cuda':
        STAGE = 'smoke'
        smoke = run_suite(OUTPUT / 'smoke' / 'openjev', WORKSPACE, projects=['openjev'], device=DEVICE, test_per_route=1, repeats=1, progress=show_progress)
        smoke_status = next(row for row in smoke['project_statuses'] if row['project_id'] == 'openjev')
        if smoke_status['execution_status'] != 'completed':
            partial = Path(shutil.make_archive(str(ROOT / 'open-jev-project-results-partial'), 'zip', OUTPUT))
            print('Smoke test failed; return this diagnostic ZIP:', partial)
            try:
                from google.colab import files
            except ImportError:
                pass
            else:
                files.download(str(partial))
            raise RuntimeError('GPU smoke test failed for openjev: ' + str(smoke_status) + '. Fix this before transferring full results.')
    STAGE = 'full'
    RUNS['openjev'] = run_suite(OUTPUT / 'openjev', WORKSPACE, projects=['openjev'], device=DEVICE, test_per_route=TEST_PER_ROUTE, repeats=REPEATS, progress=show_progress)
    print('phase timing: openjev duration_s=', round(time.perf_counter() - phase_started, 2))
    print(next(row for row in RUNS['openjev']['project_statuses'] if row['project_id'] == 'openjev'))

## Qwen3-0.6B Direct JSON Generation

Official source: https://huggingface.co/Qwen/Qwen3-0.6B

Repository commit: `c1899de289a04d12100db370d81485cdf75e47ca`

Checkpoint: `Qwen/Qwen3-0.6B` at `c1899de289a04d12100db370d81485cdf75e47ca`

Backend: Transformers AutoModelForCausalLM.generate

Hardware: GPU for comparison; CPU smoke only

Eligibility: `runnable`. Local benchmark adapter. Strict JSON schema; greedy decoding, thinking disabled, max_new_tokens=40; no retries. No measured results yet.

Official install: `python -m pip install transformers==4.57.6`

Local entry point: `python scripts/run_open_jev_suite.py --projects qwen_generative --device cuda --test-per-route 1`

First downloads are visible in the saved logs; no alternate checkpoint, backend, or proxy is substituted.

In [ ]:
if RUN_FULL_BENCHMARK:
    phase_started = time.perf_counter()
    if RUN_SMOKE_FIRST and DEVICE == 'cuda':
        STAGE = 'smoke'
        smoke = run_suite(OUTPUT / 'smoke' / 'qwen_generative', WORKSPACE, projects=['qwen_generative'], device=DEVICE, test_per_route=1, repeats=1, progress=show_progress)
        smoke_status = next(row for row in smoke['project_statuses'] if row['project_id'] == 'qwen_generative')
        if smoke_status['execution_status'] != 'completed':
            partial = Path(shutil.make_archive(str(ROOT / 'open-jev-project-results-partial'), 'zip', OUTPUT))
            print('Smoke test failed; return this diagnostic ZIP:', partial)
            try:
                from google.colab import files
            except ImportError:
                pass
            else:
                files.download(str(partial))
            raise RuntimeError('GPU smoke test failed for qwen_generative: ' + str(smoke_status) + '. Fix this before transferring full results.')
    STAGE = 'full'
    RUNS['qwen_generative'] = run_suite(OUTPUT / 'qwen_generative', WORKSPACE, projects=['qwen_generative'], device=DEVICE, test_per_route=TEST_PER_ROUTE, repeats=REPEATS, progress=show_progress)
    print('phase timing: qwen_generative duration_s=', round(time.perf_counter() - phase_started, 2))
    print(next(row for row in RUNS['qwen_generative']['project_statuses'] if row['project_id'] == 'qwen_generative'))

In [ ]:
if RUN_FULL_BENCHMARK:
    rows = [next(row for row in result['project_statuses'] if row['project_id'] == project) for project, result in RUNS.items()]
    print(json.dumps(rows, indent=2))
    print('Completed measurements and unavailable statuses remain distinct. Review raw responses, repeat agreement, and environment before using a score.')

In [ ]:
if RUN_FULL_BENCHMARK:
    from open_jev_benchmark.plots import plot_results
    plot_paths = plot_results(rows, OUTPUT / 'figures')
    print('Exported figures:', [str(path) for path in plot_paths])
    summary_path = OUTPUT / 'notebook-summary-manifest.json'
    driver_file = ROOT / 'gpu-driver-fingerprint.json'
    if driver_file.exists():
        import shutil
        shutil.copy2(driver_file, OUTPUT / driver_file.name)
    summary = {'schema_version': 'open-jev-notebook-summary/v1', 'publication_ready': False, 'bundle_sha256': BUNDLE_SHA256, 'project_statuses': rows, 'artifacts': [{'path': str(path.relative_to(OUTPUT)), 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()} for path in OUTPUT.rglob('*') if path.is_file() and path != summary_path]}
    summary_path.write_text(json.dumps(summary, indent=2))
    import shutil
    archive_path = Path(shutil.make_archive(str(ROOT / 'open-jev-project-results'), 'zip', OUTPUT))
    print('Return this result archive:', archive_path)
    print('Also save/download the executed notebook with its outputs.')
    try:
        from google.colab import files
    except ImportError:
        print('Local run: results are available at the printed path.')
    else:
        files.download(str(archive_path))

## Keep exploring

- [Open decision models article](https://www.lambertleong.com/thoughts/open-decision-models)
- [The decision-layer argument](https://www.lambertleong.com/thoughts/system-1-decision-models)
- [Build the scorer from scratch](https://www.lambertleong.com/thoughts/jev-from-scratch)

These small panels do not validate calibration, automation thresholds or performance on your own routing data. Use held-out examples from your workflow before relying on a confidence threshold.